# Gabarito — Exercício Prático · Dia 02

**Semana 05 · groupby · merge · pivot_table**

Este notebook contém a solução comentada de todos os 8 passos do exercício da Aula 02. Cada passo mostra:

- O código completo e funcional
- A explicação do raciocínio por trás de cada escolha
- A interpretação dos resultados e dos gráficos

> **Como usar este gabarito:** tente resolver o exercício sozinho primeiro. Use este arquivo só para comparar sua solução ou desbloquear um passo específico onde você ficou travado.

---
## Passo 1 — Setup: carregar dados e preparar colunas

**O que o passo pede:**
- Carregar `base_rh.xlsx` com `pd.read_excel(URL)`
- Converter `Data_Admissao` para datetime
- Criar a coluna `Ano_Admissao` com `.dt.year`

**Por que `.dt.year`?**
Quando convertemos uma coluna para o tipo `datetime`, o pandas desbloqueia o acessor `.dt`, que dá acesso a partes da data: `.dt.year`, `.dt.month`, `.dt.day`, `.dt.weekday`... Extrair o ano é o primeiro passo para qualquer análise temporal.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

URL = (
    "https://raw.githubusercontent.com/"
    "cfneves/turma-visualizacao-de-dados/"
    "master/aulas/semana_04/bases/base_rh.xlsx"
)

df = pd.read_excel(URL)

# Converte a coluna de texto para o tipo datetime
df["Data_Admissao"] = pd.to_datetime(
    df["Data_Admissao"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Extrai só o ano — facilita o groupby temporal
df["Ano_Admissao"] = df["Data_Admissao"].dt.year

print(f"Dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\nTipos das colunas novas:")
print(df[["Data_Admissao", "Ano_Admissao"]].dtypes)
print(f"\nAnos presentes: {sorted(df['Ano_Admissao'].dropna().unique().astype(int)[:].tolist())}")

---
## Passo 2 — groupby com duas colunas: Departamento × Cargo

**O que o passo pede:**
- Agrupar por `["Departamento", "Cargo"]` e contar funcionários
- Exibir com `reset_index()` e `to_string(index=False)`

**Por que duas colunas no groupby?**
Um `groupby` com lista de colunas cria um índice **hierárquico** (MultiIndex). O `reset_index()` transforma esse índice em colunas normais — essencial para exibir e plotar o resultado.

**Por que contar `ID_Funcionario` e não `.size()`?**
Ambos chegam ao mesmo resultado aqui. `.size()` conta linhas; `.count()` em uma coluna específica ignora valores nulos naquela coluna. Como `ID_Funcionario` não tem nulos, são equivalentes — mas `.count()` em uma coluna é mais explícito sobre o que está sendo contado.

In [ ]:
# Conta funcionários para cada combinação Departamento + Cargo
headcount_cargo = (
    df.groupby(["Departamento", "Cargo"])["ID_Funcionario"]
    .count()
    .reset_index()
    .rename(columns={"ID_Funcionario": "Quantidade"})
    .sort_values(["Departamento", "Quantidade"], ascending=[True, False])
)

print(f"Total de combinações Departamento × Cargo: {len(headcount_cargo)}")
print()
print(headcount_cargo.to_string(index=False))

---
## Passo 3 — Admissões por ano com filtro 2020–2024

**O que o passo pede:**
- Contar admissões por `Ano_Admissao`
- Filtrar apenas 2020 a 2024 com `query()`

**Por que `query()` aqui?**
O `query()` é mais legível que `df[(df["Ano"] >= 2020) & (df["Ano"] <= 2024)]` — especialmente quando a condição envolve um intervalo. Funciona direto no resultado do groupby depois do `reset_index()`, porque a coluna `Ano_Admissao` já está disponível como coluna normal.

**Armadilha comum:** fazer o `query()` antes do `reset_index()` gera `KeyError` porque a coluna ainda é o índice, não uma coluna acessível pelo nome.

In [ ]:
admissoes = (
    df.groupby("Ano_Admissao")["ID_Funcionario"]
    .count()
    .reset_index()                                          # Ano_Admissao vira coluna
    .rename(columns={"ID_Funcionario": "Admissoes"})
    .query("Ano_Admissao >= 2020 and Ano_Admissao <= 2024") # filtra após reset
    .sort_values("Ano_Admissao")
)

print("Admissões por ano (2020–2024):")
print(admissoes.to_string(index=False))
print(f"\nTotal de admissões no período: {admissoes['Admissoes'].sum()}")

---
## Passo 4 — Tabela de metas + merge left

**O que o passo pede:**
1. Criar uma tabela de metas de headcount por departamento com `pd.DataFrame`
2. Calcular o headcount real com `groupby`
3. Fazer `merge left` entre os dois
4. Identificar quais departamentos atingiram a meta

**Por que `how="left"`?**
O merge `left` mantém **todas as linhas da tabela da esquerda** (headcount real), mesmo que o departamento não exista na tabela de metas. Se usássemos `inner`, departamentos sem meta desapareceriam do resultado. `left` é mais seguro: você vê o que tem dado real e o que está faltando meta.

**Por que `.size()` em vez de `.count()`?**
`.size()` conta **linhas** do grupo — não precisa especificar coluna. É a forma mais direta de "quantos registros existem neste grupo".

In [ ]:
# ── Headcount real por departamento ─────────────────────────────────────
headcount_real = (
    df.groupby("Departamento")
    .size()
    .reset_index(name="Total_Real")
    .sort_values("Departamento")
)

print("Headcount real por departamento:")
print(headcount_real.to_string(index=False))

# ── Tabela de metas (criada manualmente, como se viesse de outro sistema) ──
metas_headcount = pd.DataFrame({
    "Departamento": ["Financeiro", "Logística", "Manutenção",
                     "Produção", "RH", "TI", "Vendas"],
    "Meta_Headcount": [135, 130, 125, 185, 165, 135, 125]
})

print("\nTabela de metas:")
print(metas_headcount.to_string(index=False))

# ── Merge left: mantém todos os departamentos do headcount real ───────────
comparacao_hc = pd.merge(
    headcount_real,
    metas_headcount,
    on="Departamento",
    how="left"
)

# Identifica se a meta foi atingida
comparacao_hc["Diferenca"]    = comparacao_hc["Total_Real"] - comparacao_hc["Meta_Headcount"]
comparacao_hc["Atingiu_Meta"] = comparacao_hc["Diferenca"].apply(
    lambda x: "✅ Sim" if x >= 0 else "❌ Não"
)

print("\nComparação real vs meta:")
print(comparacao_hc.to_string(index=False))

---
## Passo 5 — pivot_table: salário médio por Departamento × Gênero

**O que o passo pede:**
- Criar `pivot_table` com `Salario` médio
- `index="Departamento"`, `columns="Genero"`
- Calcular a diferença `F - M`

**Por que `fill_value=0`?**
Se alguma combinação Departamento + Gênero não existir na base, o pandas preencheria com `NaN`. Como queremos fazer cálculos (diferença `F - M`), `NaN` propagaria erro. `fill_value=0` evita isso — mas atenção: aqui sabemos que todos os departamentos têm ambos os gêneros, então é seguro.

**Interpretando a diferença `F - M`:**
- Valor positivo → mulheres ganham mais naquele departamento
- Valor negativo → homens ganham mais
- Próximo de zero → equidade salarial entre gêneros

In [ ]:
pivot_genero = pd.pivot_table(
    df,
    values   = "Salario",
    index    = "Departamento",
    columns  = "Genero",
    aggfunc  = "mean",
    fill_value = 0
).round(2)

# Diferença salarial: F - M (positivo = mulheres ganham mais)
pivot_genero["Diferenca_F_M"] = (pivot_genero["F"] - pivot_genero["M"]).round(2)

print("Salário médio por Departamento e Gênero:")
print(pivot_genero.to_string())

print(f"\nResumo da diferença salarial F - M:")
print(f"  Maior gap a favor de F : {pivot_genero['Diferenca_F_M'].max():+,.2f}  "
      f"({pivot_genero['Diferenca_F_M'].idxmax()})")
print(f"  Maior gap a favor de M : {pivot_genero['Diferenca_F_M'].min():+,.2f}  "
      f"({pivot_genero['Diferenca_F_M'].idxmin()})")
print(f"  Diferença média geral  : {pivot_genero['Diferenca_F_M'].mean():+,.2f}")

---
## Passo 6 — Gráfico de linha: admissões por ano (2020–2024)

**O que o passo pede:**
- Plotar admissões por ano com `plt.plot()`
- Usar `marker="o"` para marcar cada ponto

**Por que linha e não barras?**
O eixo X é **temporal e sequencial** — existe uma ordem e uma continuidade entre os anos. O gráfico de linha comunica essa progressão; as barras comunicariam comparação entre categorias independentes. Regra: dado temporal → linha; dado categórico → barra.

In [ ]:
plt.plot(
    admissoes["Ano_Admissao"],
    admissoes["Admissoes"],
    marker="o",
    color="steelblue",
    linewidth=2,
    markersize=8
)

# Rótulo com o valor em cima de cada ponto
for _, row in admissoes.iterrows():
    plt.text(row["Ano_Admissao"], row["Admissoes"] + 1.5,
             str(int(row["Admissoes"])),
             ha="center", va="bottom", fontsize=10)

plt.title("Admissões por Ano (2020–2024)")
plt.xlabel("Ano")
plt.ylabel("Quantidade de Admissões")
plt.xticks(admissoes["Ano_Admissao"].astype(int))
plt.ylim(0, admissoes["Admissoes"].max() * 1.15)
plt.tight_layout()
plt.show()

---
### O que este gráfico mostra?

Cada ponto representa o total de funcionários admitidos em um ano. A linha conecta os pontos e mostra a **tendência de contratações** no período de 2020 a 2024.

### Por que isso importa?

Para o gerente de produção, esse gráfico responde: **a empresa está crescendo em ritmo constante ou há variações?**

- Uma linha ascendente indica expansão contínua do quadro
- Quedas em anos específicos podem indicar congelamento de vagas, crise econômica ou mudança de estratégia de contratação
- Comparar com metas de headcount (Passo 4) permite identificar se as admissões estão alinhadas com o planejamento

---
## Passo 7 — Gráfico de barras agrupadas: salário médio por gênero

**O que o passo pede:**
- Plotar o `pivot_table` do Passo 5 com `pivot.plot(kind="bar")`
- Usar as colunas `"F"` e `"M"` (sem a coluna `Diferenca_F_M`)

**Por que `pivot_genero[["F", "M"]]` e não `pivot_genero`?**
A coluna `Diferenca_F_M` é um número calculado — se incluída, o matplotlib tentaria plotar ela também como uma terceira série, tornando o gráfico confuso. Selecionamos só as colunas que queremos visualizar.

In [ ]:
pivot_genero[["F", "M"]].plot(
    kind="bar",
    color=["#e879a0", "steelblue"],
    width=0.7
)

plt.title("Salário Médio por Gênero e Departamento")
plt.xlabel("Departamento")
plt.ylabel("Salário Médio (R$)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Gênero", labels=["Feminino", "Masculino"])
plt.tight_layout()
plt.show()

---
### O que este gráfico mostra?

Para cada departamento, duas barras lado a lado: a barra rosa mostra o **salário médio das mulheres** e a barra azul mostra o **salário médio dos homens**. Quando as duas barras têm altura parecida, os salários são equilibrados; quando uma é visivelmente mais alta, há uma diferença salarial entre gêneros.

### Por que isso importa?

Análise de equidade salarial é uma das demandas mais frequentes em gestão de pessoas:

- Diferenças salariais entre gêneros no mesmo departamento podem indicar desigualdade estrutural — dado importante para auditorias de RH e relatórios de ESG
- Departamentos com gaps grandes merecem investigação mais profunda: é diferença de cargo? de tempo de empresa? ou realmente de gênero?
- O gráfico é o ponto de partida — os números exatos do Passo 5 (`Diferenca_F_M`) são o que sustenta a análise

---
## Passo 8 — Commit no repositório

**O que o passo pede:**
Salvar o trabalho no git com a mensagem correta.

Execute no terminal (não no notebook):

```bash
git pull origin master
git add alunos/seunome/semana_05/
git commit -m "semana 05 - dia 02: groupby, merge, pivot_table, graficos"
git push
git log --oneline -3
```

**Por que `git pull` antes do commit?**
Garante que você está com a versão mais recente do repositório antes de enviar suas alterações. Se outro aluno fez push enquanto você trabalhava, o pull traz as mudanças dele primeiro — evitando conflitos na hora do push.

**Por que a mensagem de commit segue um padrão?**
Mensagens padronizadas tornam o histórico do projeto legível. Meses depois, `git log --oneline` vai mostrar claramente o que cada commit mudou sem precisar abrir o código.

---
## Resumo — O que você praticou neste exercício

| Passo | Operação | Conceito principal |
|---|---|---|
| 1 | `pd.read_excel` + `pd.to_datetime` + `.dt.year` | Carregamento e preparação |
| 2 | `groupby(["Depto", "Cargo"]).count()` | Groupby com duas colunas |
| 3 | `groupby("Ano").count()` + `.query()` | Groupby temporal + filtro |
| 4 | `pd.DataFrame` + `merge(how="left")` | Criação manual + merge left |
| 5 | `pd.pivot_table` + diferença calculada | Tabela cruzada |
| 6 | `plt.plot(marker="o")` | Gráfico de linha temporal |
| 7 | `pivot.plot(kind="bar")` | Barras agrupadas a partir de pivot |
| 8 | `git pull / add / commit / push` | Controle de versão |

**Erros mais comuns neste exercício:**

- **Passo 3:** fazer `query()` antes de `reset_index()` → `KeyError`
- **Passo 4:** usar `merge("inner")` → departamentos sem meta somem
- **Passo 5:** incluir `Diferenca_F_M` no `plot()` → terceira série indesejada
- **Passo 7:** esquecer `rotation=45` → rótulos sobrepostos no eixo X